# Slice selection: one section per stage, chosen jointly with the cross-species pairs

Which sections should the temporal and cross-species alignments use? This notebook evaluates the output of `bench/scripts/select_sections.py` and applies a decision rule fixed before the results. It reads only saved files, so it needs no GPU, caches or raw data.

**The objective** (`mgw/bench/selection.py`): over the ladder X1–X10, minimize

$$\sum_{\text{temporal}} \big(c_W + w\,|\Delta L|\big) + \lambda \sum_{\text{cross}} \big(c_W + w\,|\Delta L|\big)$$

- **c_W:** transport cost of Wasserstein OT on the organ labels, with tile masses proportional to their bins.
- **ΔL:** difference in raw laterality. L is computed from the labels too, so it reweights them by anatomy and is not independent evidence.
- **Two variants:** `w = 0` (labels only) and `w = 0.5` (labels + ΔL), both at `λ = 1`.

**The independent check:** gene r through the label coupling. No genes enter that coupling, but gene r favours spread-out couplings, so it is only used if it passes the control below.

**The rule:**
1. If gene r doesn't track the distance between sections within human embryos (median ρ < 0.3), it cannot arbitrate, and `w = 0` is selected.
2. Otherwise `w = 0.5` is selected only if its gene r gain exceeds the difference between seeds. Ties go to `w = 0`.
3. Sections that change between seeds or across the sensitivity grid are reported as uncertain.
4. Weak steps are flagged, not corrected.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
ROOT = None        # bench root; None: $MGW_BENCH_DIR or <data dir>/bench
RES = 5000         # reduce.max_obs of the run ("all": every bin)
POOL = "all"       # "all" | "series": chains restricted to the embryos of series.yaml

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

from mgw.bench import config, metrics, prepare, selection as sel, series

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 300, "display.max_colwidth", 70)
BASE = prepare.bench_root(ROOT) / "diag" / "slice_selection" / f"n{RES}"
OUT = BASE / "series" if POOL == "series" else BASE
if not (OUT / "chains.csv").exists():
    raise FileNotFoundError(f"no chains in {OUT}: run bench/scripts/select_sections.py (or copy that folder here)")

cand = pd.read_csv(OUT / "candidates.csv", dtype={"embryo": str, "stage": str}, keep_default_na=False, na_values=[""])
cand["excluded"], cand["flag"] = cand.excluded.fillna(""), cand.flag.fillna("")
chains = pd.read_csv(OUT / "chains.csv")
grid = pd.read_csv(OUT / "sensitivity.csv")
margins = pd.read_csv(OUT / "margins.csv", keep_default_na=False, na_values=[""])
ec = pd.read_csv(OUT / "edges_chain.csv")
SEEDS = sorted(int(s) for s in chains.seed.unique())
keep = set(cand.section[cand.excluded == ""])
edges = {s: (lambda e: e[e.a.isin(keep) & e.b.isin(keep)])(pd.read_csv(BASE / f"edges_seed{s}.csv")) for s in SEEDS}
control = pd.concat([pd.read_csv(OUT / f"control_seed{s}.csv") for s in SEEDS], ignore_index=True)

ser = series.load(config.load().series_path())
ladder = sel.rungs(ser, stages=set(cand.stage[cand.excluded == ""]))
H, M = sel.stage_orders(ladder)
HAS_GENES = "gene_r" in ec and ec.gene_r.notna().any()

# light surface and ink as in label_alignment_best; sequential blue; two categorical slots, fixed per variant
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SEQ = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
VARIANT = {0.0: ("#2a78d6", "o", "labels only (w = 0)"), 0.5: ("#eb6834", "s", "labels + ΔL (w = 0.5)")}


def short(s):
    return s.split("_", 1)[1]


def chain(w, seed, lam=1.0, table=grid):
    "{stage: section} of both species."
    c = table[(table.w == w) & (table.seed == seed) & (table.lam == lam)]
    return dict(zip(c.stage, c.section))


def style(ax):
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0, labelsize=8)
    for s in ax.spines.values():
        s.set_visible(False)


print(f"{OUT}\nseeds {SEEDS}; rungs {', '.join(r[0] for r in ladder)}; gene r {'present' if HAS_GENES else 'NOT computed'}")

## 1. Candidates

Sections per stage, with the reason for each exclusion. A flagged section stays a candidate.

In [ ]:
stages = H + M
t = cand[cand.stage.isin(stages)]
print(t.assign(candidate=t.excluded == "").groupby("stage", sort=False).agg(
    sections=("section", "size"), candidates=("candidate", "sum")).reindex(stages).to_string())
x = t[t.excluded != ""][["section", "excluded", "frac_classified"]]
print("\nexcluded\n" + (x.round(3).to_string(index=False) if len(x) else "none"))
f = t[t.flag != ""][["section", "flag", "excluded"]]
print("\nflagged\n" + (f.to_string(index=False) if len(f) else "none"))

## 2. The control: what does each score see of the sectioning plane?

Within a human embryo the section numbers follow the cutting order. **ρ** is the Spearman correlation of a score with the distance between two sections (per embryo; higher is better). **Hit rate** is the share of sections whose best match is their nearest available section. Scores are averaged over the seeds per section pair.

This control covers sections of one stage only, so it says nothing direct about matching across stages. The result decides whether gene r may arbitrate between the chains.

In [ ]:
SCORES = {"w_cost": "c_W (labels)", "tv": "TV of composition", "dL": "|ΔL| (raw L)", "w_cost_dL": "c_W + 0.5 |ΔL|", "gene_r": "gene r"}
cols = ["w_cost", "tv", "dL"] + (["gene_r"] if "gene_r" in control else [])
ctrl = sel.control_table(control.groupby(["kind", "a", "b", "gap"], as_index=False)[cols].mean())
if ctrl.empty:
    print("no control edges: no embryo with enough candidate sections in this run")
else:
    present = [k for k in SCORES if f"rho_{k}" in ctrl and ctrl[f"rho_{k}"].notna().any()]
    print(ctrl.round(2).to_string(index=False))
    print("\nmedian over embryos\n" + pd.DataFrame({SCORES[k]: dict(rho=ctrl[f"rho_{k}"].median(), hit=ctrl[f"hit_{k}"].median()) for k in present}).T.round(2).to_string())
    fig, ax = plt.subplots(figsize=(7, 0.55 * len(present) + 1.2), layout="constrained")
    for i, k in enumerate(present):
        ax.plot(ctrl[f"rho_{k}"], np.full(len(ctrl), i), "o", ms=8, color="#2a78d6", mec=SURFACE, mew=1.5, zorder=3)
        ax.plot([ctrl[f"rho_{k}"].median()] * 2, [i - 0.3, i + 0.3], color=INK, lw=2, zorder=4)
    if "gene_r" in present:
        ax.axvline(sel.CONTROL_RHO_MIN, color=MUTED, lw=1, ls="--")
        ax.annotate(f"gene r must reach {sel.CONTROL_RHO_MIN}", (sel.CONTROL_RHO_MIN, len(present) - 0.5), xytext=(5, 0), textcoords="offset points", fontsize=8, color=MUTED, va="center")
    ax.set_yticks(range(len(present)), [SCORES[k] for k in present], color=INK, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("ρ with the distance between sections (one dot per embryo, bar: median)", color=MUTED, fontsize=9)
    style(ax)
    ax.set_title("Which scores track the cutting order within a human embryo?", color=INK, fontsize=10, loc="left")
    plt.show()

## 3. The temporal steps

The label cost c_W of every section pair per step (seed 0), on one colour scale. Outlines mark the chain sections: solid for labels only, dashed for labels + ΔL. A row or column with similar values means the step has no clear best section.

In [ ]:
s0 = SEEDS[0]
e = edges[s0]
steps = list(zip(H, H[1:])) + list(zip(M, M[1:]))
stage_of, number = cand.set_index("section").stage, (cand.embryo + "." + cand.number.map("{:05.0f}".format)).set_axis(cand.section).to_dict()
tmp = e[e.kind == "temporal"].assign(sa=lambda d: d.a.map(stage_of), sb=lambda d: d.b.map(stage_of))
lo, hi = tmp.w_cost.min(), tmp.w_cost.max()
ch0, ch1 = chain(0.0, s0), chain(0.5, s0)
ncol = 3
nrow = -(-len(steps) // ncol) if steps else 1
fig, axes = plt.subplots(nrow, ncol, squeeze=False, figsize=(5 * ncol, 4 * nrow), layout="constrained")
for ax in axes.ravel()[len(steps):]:
    ax.set_visible(False)
for ax, (a, b) in zip(axes.ravel(), steps):
    g = tmp[(tmp.sa == a) & (tmp.sb == b)]
    v = g.pivot(index="a", columns="b", values="w_cost")
    v = v.loc[sorted(v.index, key=number.get), sorted(v.columns, key=number.get)]   # cutting order, not text order
    im = ax.imshow(v.to_numpy(), cmap=SEQ, vmin=lo, vmax=hi, aspect="auto")
    ax.set_xticks(range(v.shape[1]), [short(c) for c in v.columns], fontsize=7, rotation=90, color=INK)
    ax.set_yticks(range(v.shape[0]), [short(c) for c in v.index], fontsize=7, color=INK)
    ax.tick_params(length=0)
    for sp_ in ax.spines.values():
        sp_.set_visible(False)
    for ch, ls in ((ch0, "-"), (ch1, "--")):
        ax.add_patch(Rectangle((list(v.columns).index(ch[b]) - .5, list(v.index).index(ch[a]) - .5), 1, 1, fill=False, ec=INK, lw=2, ls=ls))
    ax.set_title(f"{a} → {b}", fontsize=9, color=INK)
fig.colorbar(im, ax=axes, shrink=0.5, label="c_W (label cost)")
plt.show()

## 4. The chains compared

Per step and variant (seed 0). `bin_ratio` is the later section's tissue bins over the earlier one's, or mouse over human on cross edges. `w_cost` and `dL` are label-derived; `gene_r` is not.

In [ ]:
show = ["kind", "a", "b", "w_cost", "tv", "dL", "K", "bin_ratio"] + (["gene_r", "n_genes"] if HAS_GENES else [])
for w in sel.MAIN_W:
    c = chain(w, s0)
    print(f"--- {VARIANT[w][2]}\n  human: {' → '.join(c[s] for s in H)}\n  mouse: {' → '.join(c[s] for s in M)}")
    print(sel.weak_steps(ec[(ec.w == w) & (ec.seed == s0)])[show + ["large_dL"] + (["low_gene_r"] if HAS_GENES else [])].round(3).to_string(index=False), "\n")
print("sections that differ between the variants, per seed")
for s in SEEDS:
    a, b = chain(0.0, s), chain(0.5, s)
    print(f"  seed {s}: " + (", ".join(f"{a[k]} / {b[k]}" for k in a if a[k] != b[k]) or "none"))

In [ ]:
metrics_ = {"w_cost": "c_W (labels)", "dL": "|ΔL|"}
if HAS_GENES:
    metrics_["gene_r"] = "gene r"
kinds = {"human, temporal": ("temporal", H), "mouse, temporal": ("temporal", M), "cross": ("cross", None)}
fig, axes = plt.subplots(len(metrics_), len(kinds), squeeze=False, sharey="row", figsize=(4.4 * len(kinds), 2.1 * len(metrics_) + 1), layout="constrained")
for j, (title, (kind, order)) in enumerate(kinds.items()):
    for i, (col, name) in enumerate(metrics_.items()):
        ax = axes[i, j]
        for w, (colour, marker, _) in VARIANT.items():
            c = chain(w, s0)
            q = ec[(ec.w == w) & (ec.seed == s0)].set_index(["a", "b"])[col]
            if order is None:
                labels = [r[0] for r in ladder]
                y = [q[c[hs], c[ms]] for _, hs, ms in ladder]
            else:
                labels = [f"{a}→{b}" for a, b in zip(order, order[1:])]
                y = [q[c[a], c[b]] for a, b in zip(order, order[1:])]
            ax.plot(range(len(y)), y, color=colour, lw=2, marker=marker, ms=8, mec=SURFACE, mew=1.5, label=VARIANT[w][2])
        ax.set_xticks(range(len(labels)), labels, rotation=45 if order else 0, ha="right" if order else "center")
        style(ax)
        if i == 0:
            ax.set_title(title, fontsize=10, color=INK)
        if j == 0:
            ax.set_ylabel(name, color=INK)
fig.legend(handles=[Line2D([], [], color=v[0], marker=v[1], ms=8, mec=SURFACE, lw=2, label=v[2]) for v in VARIANT.values()],
           loc="outside lower center", ncol=2, frameon=False, labelcolor=INK)
plt.show()

## 5. How stable is the choice?

**Sensitivity:** per stage, the share of all (w, λ, seed) settings that pick each section. `λ = 0` is the independent chains. **Margin:** how much the objective rises without a section, next to the section that replaces it; small margins mean a near tie. The margins come from the main variants.

In [ ]:
n = grid.groupby(["w", "lam", "seed"]).ngroups
share = grid.groupby(["species", "stage", "section"], sort=False).size().div(n).rename("share").reset_index()
for sp_, order in (("human", H), ("mouse", M)):
    print(sp_)
    for st in order:
        g = share[share.stage == st].sort_values("share", ascending=False)
        print(f"  {st:8s} " + ", ".join(f"{short(r.section)} {r.share:.0%}" for r in g.itertuples()))
print()
print(margins[margins.seed == s0].pivot_table(index=["species", "stage", "section"], columns="w", values="margin", sort=False).round(3).to_string())
print()
print(margins[margins.seed == s0][["w", "stage", "section", "margin", "alternative"]].round(3).to_string(index=False))

## 6. Organ maps of the chains

Each chain in stage order, seed 0 tiles, to check by eye that the sections look comparable. **Spinal cord** is always coloured, since it indicates a section near the midline; the seven other largest organs follow in fixed colours, and the rest are grey. Sections are in their stored orientation and scaled per panel, so sizes are not comparable.

In [ ]:
view = np.load(OUT / "view.npz", allow_pickle=False)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
REST = "#d6d5d0"
shown = sorted({s for w in sel.MAIN_W for s in chain(w, s0).values()})
area = np.sum([np.bincount(view[f"{s}|label"], minlength=len(metrics.ORGANS)) for s in shown], axis=0).astype(float)
area[~metrics.SCORED] = 0
sc = metrics.ORGAN_INDEX["Spinal cord"]
top = [sc] + [i for i in np.argsort(-area) if i != sc and area[i] > 0][:7]
LUT = np.array([REST] * len(metrics.ORGANS), dtype=object)
for i, c in zip(top, SLOTS):
    LUT[i] = c


def organ_row(axes, secs, title):
    for ax, (st, s) in zip(axes, secs):
        xy, lab = view[f"{s}|xy"], view[f"{s}|label"]
        ax.scatter(xy[:, 0], xy[:, 1], c=list(LUT[lab]), s=2, linewidths=0, rasterized=True)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.axis("off")
        ax.set_title(f"{st}\n{short(s)}", fontsize=8, color=INK)
    axes[0].annotate(title, (0, 1.22), xycoords="axes fraction", fontsize=9, color=MUTED)


rows = [(w, name, order) for w in sel.MAIN_W for name, order in (("human", H), ("mouse", M))]
nc = max(len(H), len(M))
fig, axes = plt.subplots(len(rows), nc, squeeze=False, figsize=(2.1 * nc, 2.7 * len(rows)), layout="constrained")
for ax in axes.ravel():
    ax.axis("off")
for i, (w, name, order) in enumerate(rows):
    c = chain(w, s0)
    organ_row(axes[i], [(st, c[st]) for st in order], f"{name}, {VARIANT[w][2]}")
fig.legend(handles=[Patch(color=c, label=metrics.ORGANS[i]) for i, c in zip(top, SLOTS)] + [Patch(color=REST, label="other organs, Other")],
           loc="outside lower center", ncol=5, frameon=False, fontsize=9, labelcolor=INK)
plt.show()

## 7. Decision

The rule from the top, applied to these results. The output is the proposed `bench/configs/chains.yaml`. It is printed, not written: commit it only after reviewing this notebook.

In [ ]:
sc_ = ec.groupby(["w", "seed"], as_index=False).gene_r.mean() if HAS_GENES else pd.DataFrame(columns=["w", "seed", "gene_r"])
d = sel.decide(ctrl if HAS_GENES else pd.DataFrame(), sc_)
if HAS_GENES:
    print(sc_.pivot(index="seed", columns="w", values="gene_r").round(4).to_string(), "\n")
print(f"decision: w = {d['w']}\nreason: {d['reason']}")
final = chain(d["w"], s0)
uncertain = []
for st, s in final.items():
    other = grid[(grid.stage == st) & (grid.lam > 0) & (grid.w == d["w"])]
    if set(other.section) != {s}:
        uncertain.append(f"{s} (also {', '.join(sorted(set(other.section) - {s}))})")
print("\nuncertain sections (changing with the seed or λ > 0 at this w): " + ("; ".join(uncertain) or "none"))
w_ec = sel.weak_steps(ec[(ec.w == d["w"]) & (ec.seed == s0)])
weak = w_ec[w_ec.large_dL | (w_ec.low_gene_r if HAS_GENES else False)]
print(f"\nflagged steps (|ΔL| > {sel.DL_WEAK}" + (", or lowest quarter of gene r per kind" if HAS_GENES else "") + ")")
print(weak[["kind", "a", "b", "dL"] + (["gene_r", "low_gene_r"] if HAS_GENES else [])].round(3).to_string(index=False) if len(weak) else "none")
print(f"\n# bench/configs/chains.yaml (w = {d['w']}, lambda = 1, seed {s0}, max_obs {RES}, pool {POOL})\nhuman: [{', '.join(final[s] for s in H)}]\nmouse: [{', '.join(final[s] for s in M)}]")